# KTTS XTTS-v2 adaptation v1
Adapts pretrained XTTS-v2 to the clean single-speaker KTTS corpus using deterministic tokenizer-safe Kashmiri ASCII G2P. Non-commercial experiment; human listening review is required.


In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher
!pip install -q --no-cache-dir 'coqui-tts==0.28.0' 'coqui-tts-trainer>=0.4.0,<0.5.0' pyarrow soundfile scipy
!pip uninstall -y -q tensorboard tensorflow jax jaxlib
!pip install -q --no-cache-dir 'tensorboard==2.19.0'
!python -c "from TTS.tts.configs.xtts_config import XttsAudioConfig; from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTTrainer, GPTTrainerConfig; from trainer import Trainer; print('XTTS trainer ready')"


In [ ]:
!python /kaggle/working/cipher/scripts/prepare-ktts-matcha.py --repo-dir /kaggle/working/cipher --output /kaggle/working/ktts-xtts --local-root /kaggle/input
from pathlib import Path
import json
report=json.loads(Path('/kaggle/working/ktts-xtts/preparation-report.json').read_text())
print(report)
assert report['accepted'] >= 2500 and report['hours'] >= 3.5
# Prepared WAVs are self-contained; remove only redundant HF dataset caches.
import shutil
for cache_root in (Path('/root/.cache/huggingface/datasets'), Path('/root/.cache/huggingface/hub')):
    if cache_root.is_dir():
        for cached in cache_root.glob('*kashmiri-tts-single-speaker*'):
            shutil.rmtree(cached) if cached.is_dir() else cached.unlink()
!df -h /kaggle/working


In [ ]:
import os, subprocess, sys
from pathlib import Path
env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['PYTHONUNBUFFERED']='1'; env['XTTS_EPOCHS']='15'
log=Path('/kaggle/working/xtts-training.log')
cmd=[sys.executable,'-u','/kaggle/working/cipher/scripts/train-ktts-xtts.py']
with log.open('w') as handle: result=subprocess.run(cmd,env=env,stdout=handle,stderr=subprocess.STDOUT,text=True,timeout=41400)
print('Training exit code:',result.returncode); print(log.read_text(errors='replace')[-30000:])
if result.returncode != 0: raise RuntimeError('XTTS training failed; inspect xtts-training.log')


In [ ]:
from pathlib import Path
artifact=Path('/kaggle/working/ktts-xtts-v1.zip')
assert artifact.is_file() and artifact.stat().st_size > 1_000_000
print('Artifact:',artifact,'bytes:',artifact.stat().st_size)
